# Regularization Methods — Exercises

This notebook contains 8 exercises covering shrinkage, $L_1$ sparsity,
dropout expectation, early stopping, mixup, spectral normalization, AdamW,
and practical regularization design.

**Difficulty:** Exercises 1-3 are mechanical, 4-6 are conceptual and
derivational, and 7-8 are AI-facing design exercises.


In [ ]:
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print("  value :", value)
        print("  target:", target)
    return ok

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

print("Exercise helpers ready.")


---

## Exercise 1 [*] — Ridge update and shrinkage

For

$$
\mathcal{J}(oldsymbol{	heta}) = \mathcal{L}_{	ext{data}}(oldsymbol{	heta}) + rac{\lambda}{2}\lVert oldsymbol{	heta}Vert_2^2,
$$

write the one-step SGD update when the data gradient is $\mathbf{g}$.
Show that it can be rewritten as a multiplicative shrinkage plus the data step.


In [ ]:
# Exercise 1: Your Solution
theta = np.array([1.0, -2.0, 0.5])
g = np.array([0.3, -0.4, 0.2])
eta = 0.1
lam = 0.2
pass


In [ ]:
# Exercise 1: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

header("Exercise 1: Ridge shrinkage update")
theta = np.array([1.0, -2.0, 0.5])
g = np.array([0.3, -0.4, 0.2])
eta = 0.1
lam = 0.2
coupled = theta - eta * (g + lam * theta)
decayed = (1.0 - eta * lam) * theta - eta * g
print("Coupled form :", coupled)
print("Decay form   :", decayed)
check_close("The two SGD forms match", coupled, decayed)
print("\nTakeaway: in vanilla SGD, an $L_2$ penalty is equivalent to multiplicative weight decay.")


---

## Exercise 2 [*] — Soft thresholding

Solve

$$
\min_{	heta} rac{1}{2}(	heta - z)^2 + \lambda |	heta|
$$

for the scalar inputs $z = -0.2, 0.3, 1.5$ with $\lambda = 0.5$.


In [ ]:
# Exercise 2: Your Solution
zs = np.array([-0.2, 0.3, 1.5])
lam = 0.5
pass


In [ ]:
# Exercise 2: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

def soft_threshold(z, lam):
    z = np.asarray(z)
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0.0)

header("Exercise 2: Soft thresholding")
zs = np.array([-0.2, 0.3, 1.5])
lam = 0.5
answer = soft_threshold(zs, lam)
print("Answer:", answer)
check_close("Correct proximal output", answer, np.array([0.0, 0.0, 1.0]))
print("\nTakeaway: $L_1$ both shrinks and creates exact zeros through the threshold at $\\lambda$.")


---

## Exercise 3 [*] — Inverted dropout expectation

Let

$$
	ilde{h} = rac{m}{q} h,
\qquad
m \sim \operatorname{Bern}(q).
$$

Compute $\mathbb{E}[	ilde{h}]$ and $\operatorname{Var}(	ilde{h})$ for
$h = 2$ and $q = 0.8$.


In [ ]:
# Exercise 3: Your Solution
h = 2.0
q = 0.8
pass


In [ ]:
# Exercise 3: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

header("Exercise 3: Inverted dropout moments")
h = 2.0
q = 0.8
mean = h
var = h**2 * (1.0 - q) / q
print("Mean    :", mean)
print("Variance:", var)
check_close("Expectation is preserved", mean, 2.0)
check_close("Variance formula is correct", var, 1.0)
print("\nTakeaway: inverted dropout keeps the mean fixed while injecting variance that grows as keep probability falls.")


---

## Exercise 4 [**] — Early-stopping fit fractions

For singular values $\sigma = [0.5, 2.0, 5.0]$, learning rate $\eta = 0.05$,
and step count $t = 40$, compute the early-stopping fit fraction

$$
f_t(\sigma) = 1 - (1 - \eta \sigma^2)^t.
$$

Which direction is fitted most slowly?


In [ ]:
# Exercise 4: Your Solution
sigmas = np.array([0.5, 2.0, 5.0])
eta = 0.05
t = 40
pass


In [ ]:
# Exercise 4: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

header("Exercise 4: Early-stopping fit fraction")
sigmas = np.array([0.5, 2.0, 5.0])
eta = 0.05
t = 40
frac = 1.0 - (1.0 - eta * sigmas**2) ** t
print("Fit fractions:", np.round(frac, 4))
slowest = sigmas[np.argmin(frac)]
print("Most slowly fitted singular value:", float(slowest))
check_true("The smallest singular value is the most damped", slowest == 0.5)
print("\nTakeaway: early stopping suppresses fragile low-signal directions because they accumulate slowly along the optimization path.")


---

## Exercise 5 [**] — Mixup produces valid soft labels

Suppose

$$
\mathbf{y}_i = [1, 0, 0], \qquad \mathbf{y}_j = [0, 1, 0], \qquad \lambda = 0.3.
$$

Compute the mixup target

$$
	ilde{\mathbf{y}} = \lambda \mathbf{y}_i + (1-\lambda)\mathbf{y}_j.
$$


In [ ]:
# Exercise 5: Your Solution
y_i = np.array([1.0, 0.0, 0.0])
y_j = np.array([0.0, 1.0, 0.0])
lam = 0.3
pass


In [ ]:
# Exercise 5: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

header("Exercise 5: Mixup soft target")
y_i = np.array([1.0, 0.0, 0.0])
y_j = np.array([0.0, 1.0, 0.0])
lam = 0.3
y_mix = lam * y_i + (1.0 - lam) * y_j
print("Mixed target:", y_mix)
check_close("Correct mixed label", y_mix, np.array([0.3, 0.7, 0.0]))
check_true("The result is still a probability distribution", abs(y_mix.sum() - 1.0) < 1e-12)
print("\nTakeaway: mixup regularizes both the input geometry and the supervision signal.")


---

## Exercise 6 [**] — Spectral normalization by power iteration

Estimate the top singular value of

$$
W =
egin{bmatrix}
3 & 0 \
4 & 0
\end{bmatrix}
$$

and normalize the matrix so the spectral norm is $1$.


In [ ]:
# Exercise 6: Your Solution
W = np.array([[3.0, 0.0], [4.0, 0.0]])
pass


In [ ]:
# Exercise 6: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_close(name, value, target, tol=1e-8):
    ok = np.allclose(value, target, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    return ok

header("Exercise 6: Spectral normalization")
W = np.array([[3.0, 0.0], [4.0, 0.0]])
sigma = np.linalg.svd(W, compute_uv=False)[0]
W_sn = W / sigma
sigma_after = np.linalg.svd(W_sn, compute_uv=False)[0]
print("sigma_max before:", sigma)
print("sigma_max after :", sigma_after)
check_close("Top singular value is 5 before normalization", sigma, 5.0)
check_close("Normalized matrix has spectral norm 1", sigma_after, 1.0)
print("\nTakeaway: spectral normalization controls worst-case amplification by rescaling with the top singular value.")


---

## Exercise 7 [***] — AdamW differs from coupled $L_2$

On the toy parameter vector $oldsymbol{	heta} = [2.0, 0.2]$ with data
gradient $\mathbf{g} = [0.1, -0.8]$, compare:

1. Adam-style coupled $L_2$ using the total gradient $\mathbf{g} + \lambda oldsymbol{	heta}$
2. AdamW-style decoupled decay

Use $\lambda = 0.2$ and learning rate $0.1$.


In [ ]:
# Exercise 7: Your Solution
theta = np.array([2.0, 0.2])
g = np.array([0.1, -0.8])
lr = 0.1
lam = 0.2
pass


In [ ]:
# Exercise 7: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

header("Exercise 7: AdamW vs coupled L2")
theta = np.array([2.0, 0.2])
g = np.array([0.1, -0.8])
lr = 0.1
lam = 0.2
eps = 1e-8

g_total = g + lam * theta
theta_coupled = theta - lr * g_total / (np.sqrt(g_total**2) + eps)
theta_adamw = theta - lr * g / (np.sqrt(g**2) + eps) - lr * lam * theta
print("Coupled update:", theta_coupled)
print("AdamW update  :", theta_adamw)
check_true("The two adaptive updates differ", np.linalg.norm(theta_coupled - theta_adamw) > 1e-6)
print("\nTakeaway: adaptive preconditioning changes the meaning of a coupled $L_2$ term, so AdamW decouples decay explicitly.")


---

## Exercise 8 [***] — Regularization plan for a small-data LoRA fine-tune

You are fine-tuning a pretrained LLM with LoRA on a small supervised dataset.
Partition the following parameter names into `decay` and `no_decay` groups, then
suggest two extra regularization choices besides weight decay.

Parameters:

- `layers.0.attn.q_proj.weight`
- `layers.0.attn.q_proj.bias`
- `layers.0.input_layernorm.weight`
- `lm_head.weight`


In [ ]:
# Exercise 8: Your Solution
names = [
    "layers.0.attn.q_proj.weight",
    "layers.0.attn.q_proj.bias",
    "layers.0.input_layernorm.weight",
    "lm_head.weight",
]
pass


In [ ]:
# Exercise 8: Solution
import numpy as np

def header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

def check_true(name, condition):
    print(f"{'PASS' if condition else 'FAIL'} - {name}")
    return condition

header("Exercise 8: LoRA fine-tuning regularization plan")
names = [
    "layers.0.attn.q_proj.weight",
    "layers.0.attn.q_proj.bias",
    "layers.0.input_layernorm.weight",
    "lm_head.weight",
]

decay = []
no_decay = []
for name in names:
    if name.endswith(".bias") or "layernorm" in name or "norm" in name:
        no_decay.append(name)
    else:
        decay.append(name)

extras = ["early stopping", "LoRA dropout"]
print("Decay group   :", decay)
print("No-decay group:", no_decay)
print("Extra choices :", extras)
check_true("Bias is excluded from decay", any(name.endswith(".bias") for name in no_decay))
check_true("LayerNorm is excluded from decay", any("layernorm" in name for name in no_decay))
check_true("Two extra regularizers are proposed", len(extras) == 2)
print("\nTakeaway: for LLM fine-tuning, regularization is usually a package: selective decay plus path or stochastic controls aligned to the data regime.")
